# 03 — Live Paper Trading Session Viewer

This notebook provides a daily dashboard for the live paper trading engine.

**Sections:**
1. Today's Schedule — upcoming trot races
2. Today's Recommendations — EV+ bets from `generate_bets()`
3. Yesterday's Results — `resolve_bets()` output and daily P&L
4. 7-Day Ledger — cumulative P&L chart
5. Model Calibration — predicted probability vs actual hit rate

In [ ]:
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from config.settings import DB_PATH
from src.scraper.storage import get_connection
from src.trading.engine import compute_today_features, generate_bets, resolve_bets, get_ledger
from src.model.scorer import score_combined

conn = get_connection()
TODAY = datetime.today().strftime('%Y%m%d')
YESTERDAY = (datetime.today() - timedelta(days=1)).strftime('%Y%m%d')
print(f'Today: {TODAY}   Yesterday: {YESTERDAY}')

## 1. Today's Schedule

Upcoming trot races for today (finish_position still NULL — races haven't run).

In [ ]:
schedule_sql = """
    SELECT
        ra.race_id,
        ra.hippodrome,
        ra.race_datetime,
        ra.distance_metres,
        ra.field_size,
        COUNT(ru.runner_id) AS runners_loaded
    FROM races ra
    LEFT JOIN runners ru ON ru.race_id = ra.race_id AND ru.scratch = FALSE
    WHERE ra.date = ?
      AND ra.is_trot = TRUE
    GROUP BY ra.race_id, ra.hippodrome, ra.race_datetime, ra.distance_metres, ra.field_size
    ORDER BY ra.race_datetime
"""
schedule = conn.execute(schedule_sql, [TODAY]).df()

if schedule.empty:
    print(f'No trot races found for {TODAY}. Run morning session first:')
    print('  from src.trading.scheduler import run_morning_session')
    print('  run_morning_session()')
else:
    print(f'{len(schedule)} trot races scheduled for {TODAY}')
    display(schedule)

## 2. Today's Recommendations

EV+ bets generated by the model. Runs `generate_bets()` (idempotent — safe to re-run).

In [ ]:
bets_today = generate_bets(conn, TODAY, scorer_fn=score_combined)

if not bets_today:
    print('No EV+ bets found for today.')
else:
    df_bets = pd.DataFrame(bets_today)
    display_cols = [
        'race_id', 'hippodrome', 'bet_type',
        'horse_name_1', 'horse_name_2',
        'morning_odds', 'model_prob', 'implied_prob', 'ev_ratio',
        'kelly_stake', 'stake',
    ]
    available = [c for c in display_cols if c in df_bets.columns]
    print(f'{len(df_bets)} EV+ bets for {TODAY}:')
    display(
        df_bets[available].style
        .format({'model_prob': '{:.1%}', 'implied_prob': '{:.1%}',
                 'ev_ratio': '{:.2f}', 'morning_odds': '{:.1f}',
                 'kelly_stake': '{:.2f}', 'stake': '{:.2f}'})
    )

## 3. Yesterday's Results

Resolve pending bets from yesterday using actual finish positions.

In [ ]:
summary = resolve_bets(conn, YESTERDAY)

if summary.empty:
    print(f'No pending bets to resolve for {YESTERDAY}.')
    print('(Either no bets were placed, or results are not yet in the DB.)')
else:
    row = summary.iloc[0]
    print(f"Date:         {row['date']}")
    print(f"Bets placed:  {int(row['n_bets'])}")
    print(f"Bets won:     {int(row['n_won'])}")
    print(f"Total stake:  {row['total_stake']:.2f} €")
    print(f"Total P&L:    {row['total_pnl']:+.2f} €")
    print(f"ROI:          {row['roi']:.1%}")

# Show the resolved bets
yesterday_bets = conn.execute(
    "SELECT bet_id, bet_type, horse_name_1, horse_name_2, morning_odds, "
    "model_prob, status, stake, pnl FROM bets WHERE date = ? ORDER BY race_id",
    [YESTERDAY]
).df()

if not yesterday_bets.empty:
    display(yesterday_bets)

## 4. 7-Day Ledger

Cumulative P&L over the last 7 days of paper trading.

In [ ]:
week_ago = (datetime.today() - timedelta(days=7)).strftime('%Y%m%d')
ledger = get_ledger(conn, start_date=week_ago, end_date=TODAY)

if ledger.empty:
    print('No bets in the 7-day ledger yet.')
else:
    # Summary table by day
    daily = (
        ledger[ledger['status'].isin(['won', 'lost'])]
        .groupby('date')
        .agg(n_bets=('bet_id', 'count'),
             n_won=('status', lambda s: (s == 'won').sum()),
             total_stake=('stake', 'sum'),
             daily_pnl=('pnl', 'sum'))
        .reset_index()
    )
    daily['roi'] = daily['daily_pnl'] / daily['total_stake']
    daily['cumulative_pnl'] = daily['daily_pnl'].cumsum()

    print(f'7-day ledger: {len(ledger)} total bets')
    display(
        daily.style.format({
            'total_stake': '{:.2f}', 'daily_pnl': '{:+.2f}',
            'roi': '{:.1%}', 'cumulative_pnl': '{:+.2f}',
        })
    )

    # Cumulative P&L chart
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.step(daily['date'], daily['cumulative_pnl'], where='post', linewidth=2, color='steelblue')
    ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
    ax.fill_between(daily['date'], daily['cumulative_pnl'], 0,
                    where=(daily['cumulative_pnl'] >= 0),
                    alpha=0.2, color='green', step='post', label='Profit')
    ax.fill_between(daily['date'], daily['cumulative_pnl'], 0,
                    where=(daily['cumulative_pnl'] < 0),
                    alpha=0.2, color='red', step='post', label='Loss')
    ax.set_title('Cumulative P&L — 7-Day Rolling (paper trading)', fontsize=13)
    ax.set_xlabel('Date')
    ax.set_ylabel('P&L (€)')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%+.1f €'))
    ax.legend()
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 5. Model Calibration

Bucketed model probability vs actual win rate — checks whether the model's confidence is well-calibrated.

In [ ]:
all_bets = get_ledger(conn)
resolved = all_bets[all_bets['status'].isin(['won', 'lost'])].copy()

if resolved.empty or 'model_prob' not in resolved.columns:
    print('Not enough resolved bets for calibration (need at least one won/lost bet).')
else:
    resolved = resolved.dropna(subset=['model_prob'])
    resolved['hit'] = (resolved['status'] == 'won').astype(int)

    # Bin by model_prob into 5 buckets
    resolved['prob_bucket'] = pd.cut(
        resolved['model_prob'],
        bins=[0, 0.2, 0.3, 0.4, 0.5, 1.0],
        labels=['0–20%', '20–30%', '30–40%', '40–50%', '50%+']
    )
    calib = (
        resolved.groupby('prob_bucket', observed=True)
        .agg(
            n_bets=('hit', 'count'),
            actual_hit_rate=('hit', 'mean'),
            avg_model_prob=('model_prob', 'mean'),
        )
        .reset_index()
    )
    print('Calibration table:')
    display(calib.style.format({
        'actual_hit_rate': '{:.1%}',
        'avg_model_prob': '{:.1%}',
    }))

    # Calibration chart
    fig, ax = plt.subplots(figsize=(7, 5))
    calib_valid = calib.dropna(subset=['actual_hit_rate', 'avg_model_prob'])
    ax.scatter(calib_valid['avg_model_prob'], calib_valid['actual_hit_rate'],
               s=calib_valid['n_bets'] * 20, color='steelblue', zorder=5,
               label='Buckets (size ∝ n_bets)')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration')
    ax.set_xlim(0, 0.8)
    ax.set_ylim(0, 0.8)
    ax.set_xlabel('Average model probability')
    ax.set_ylabel('Actual hit rate')
    ax.set_title('Model Calibration (win bets)', fontsize=13)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend()
    plt.tight_layout()
    plt.show()

conn.close()
print('\nDone.')